# Student Success Prediction & Retention Strategy

## Overview
In this project, I analyzed student engagement and performance data to understand the key factors that influence course completion in an online learning environment.

The goal is to identify students at risk of dropping out early and provide actionable insights that can help improve retention.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Load data
path = '/content/online_course_engagement_data.csv'
df = pd.read_csv(path)

df.head()

## Data Understanding

Check basic structure, data types, and missing values.

In [ ]:
df.info()
df.describe()
df.isna().sum()

## Target Variable

Create a binary target variable `dropout`:
- 1 → student dropped out
- 0 → student completed

In [3]:
df["dropout"] = np.where(df["CourseCompletion"] == 0, 1, 0)

df[["CourseCompletion", "dropout"]].head()

,CourseCompletion,dropout
0,0,1
1,0,1
2,1,0
3,1,0
4,0,1


## Exploratory Data Analysis

Analyze how engagement and performance differ between students who complete courses and those who drop out.

In [6]:
#CHECKING ENGAGEMENT
df.groupby("dropout")["TimeSpentOnCourse"].mean()


,TimeSpentOnCourse
dropout,
0,56.581104
1,45.948642


In [5]:
df.groupby("dropout")["NumberOfVideosWatched"].mean()

,NumberOfVideosWatched
dropout,
0,11.768217
1,8.879418


In [7]:
#PERFORMANCE CHECKING
df.groupby("dropout")["QuizScores"].mean()

,QuizScores
dropout,
0,80.027720
1,71.210483


In [8]:
#CHECKING ACTIVITY
df.groupby("dropout")["NumberOfQuizzesTaken"].mean()

,NumberOfQuizzesTaken
dropout,
0,6.198991
1,4.362482


In [9]:
#CHECKING DEVICES USED DURING LEARNING BEHAVIOR
pd.crosstab(df["DeviceType"], df["dropout"], normalize="index")

dropout,0,1
DeviceType,,
0,0.392746,0.607254
1,0.400133,0.599867


In [ ]:
# VISUALIZING SOME RELATIONSHIPS BETWEEN VARIABLES
df.groupby("dropout")["TimeSpentOnCourse"].mean().plot(kind="bar", title="Time Spent vs Dropout")
plt.show()

df.groupby("dropout")["QuizScores"].mean().plot(kind="bar", title="Quiz Score vs Dropout")
plt.show()

df.groupby("dropout")["NumberOfVideosWatched"].mean().plot(kind="bar", title="Videos Watched vs Dropout")
plt.show()

## Key Insights

- Students who spend less time on course content are more likely to drop out  
- Lower quiz scores are associated with higher dropout risk  
- Students who watch fewer videos tend to have lower completion outcomes  
- Engagement behavior is a strong indicator of student success  

## Modeling - Logistic Regression

Build a classification model to predict student dropout.

In [11]:
# Features and target
X = df.drop(columns=["UserID", "CourseCompletion", "dropout"])
y = df["dropout"]

categorical_features = ["CourseCategory", "DeviceType"]
numeric_features = [
    "TimeSpentOnCourse",
    "NumberOfVideosWatched",
    "NumberOfQuizzesTaken",
    "QuizScores",
    "CompletionRate"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7911111111111111
[[506 208]
 [168 918]]
              precision    recall  f1-score   support

           0       0.75      0.71      0.73       714
           1       0.82      0.85      0.83      1086

    accuracy                           0.79      1800
   macro avg       0.78      0.78      0.78      1800
weighted avg       0.79      0.79      0.79      1800



## Model Improvement (Removing CompletionRate)

CompletionRate is closely related to the target and may introduce leakage.  
We remove it to build a more realistic model.

In [14]:
#Model without CompletionRate
# Features and target
X = df.drop(columns=["UserID", "CourseCompletion", "dropout", "CompletionRate"])
y = df["dropout"]

# Separate column types
categorical_features = ["CourseCategory", "DeviceType"]
numeric_features = [
    "TimeSpentOnCourse",
    "NumberOfVideosWatched",
    "NumberOfQuizzesTaken",
    "QuizScores"
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ]
)

# Pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.745

Confusion Matrix:
 [[441 273]
 [186 900]]

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.62      0.66       714
           1       0.77      0.83      0.80      1086

    accuracy                           0.74      1800
   macro avg       0.74      0.72      0.73      1800
weighted avg       0.74      0.74      0.74      1800



In [15]:
#CHECKING FEATURE IMPORTANCE
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["classifier"].coef_[0]

importance = pd.DataFrame({
    "feature": feature_names,
    "importance": coefficients
}).sort_values(by="importance", key=abs, ascending=False)

importance.head(10)

,feature,importance
3,num__QuizScores,-0.839182
2,num__NumberOfQuizzesTaken,-0.797607
1,num__NumberOfVideosWatched,-0.694974
0,num__TimeSpentOnCourse,-0.516407
5,cat__CourseCategory_Health,0.229917
4,cat__CourseCategory_Business,0.102472
6,cat__CourseCategory_Programming,0.080631
7,cat__CourseCategory_Science,0.073732
8,cat__DeviceType_1,-0.050815


## Recommendations

- Monitor early engagement (first 1–2 weeks)
- Provide support for students with low quiz scores
- Encourage consistent participation in quizzes and videos
- Use engagement metrics as early indicators of dropout risk

## Conclusion

This project shows that student engagement and performance are strong predictors of course completion.
Even after removing features closely tied to the outcome, the model maintained strong performance, indicating that behavioral data can effectively identify at-risk students early.This approach can help edtech platforms improve retention through targeted, data-driven interventions.